In [ ]:
import os
from pathlib import Path
import tempfile
import shutil

import teehr

from teehr import DeterministicMetrics as dm
from teehr import Signatures as s
from teehr import RowLevelCalculatedFields as rcf
from teehr import TimeseriesAwareCalculatedFields as tcf

from teehr.querying.utils import join_geometry

import holoviews as hv
import geoviews as gv
import hvplot.pandas
import cartopy.crs as ccrs
from holoviews import opts

teehr.__version__

In [ ]:
from teehr.evaluation.spark_session_utils import create_spark_session
# Local: Just run on the Jupyter Notebook instance
spark = create_spark_session()

# Spark executors: The nodes that the Spark executors run on have a 1 core to 8G memeory ratio.  
# It makes sesne to maintain that ratio in the executors.
# An executor with 4 cores and 32G memeory costs about $0.10/hr > this cluster costs about $9.60/hr.
# 3 of these fit on a single node so it makes sense to start increments of 3 (i.e., 3, 6, 9, ..., 96).
# This is on top of the $1.00/hr for the Jupyter Notebook instance.
# spark = create_spark_session(
#     start_spark_cluster=True,
#     executor_instances=96,
#     executor_memory="32g",
#     executor_cores=4,
# )
ev = teehr.RemoteReadOnlyEvaluation(spark=spark)

In [ ]:
SIM_METRICS = [
        s.Count(),
        # s.Average(),
        # s.Variance(),
        dm.RelativeBias(add_epsilon=True),
        # dm.NashSutcliffeEfficiency(add_epsilon=True),
        # dm.KlingGuptaEfficiency(add_epsilon=True),
        # dm.RootMeanStandardDeviationRatio(add_epsilon=True),
        dm.MeanAbsoluteError(),
        # dm.MeanAbsoluteRelativeError()
]
SIM_GROUPBY = ["primary_location_id", "configuration_name", "variable_name", "unit_name"]

In [ ]:
sdf = ev.table(
    table_name="sim_joined_timeseries"
# ).filter(
#     [
#         {
#             "column": "primary_location_id",
#             "operator": "=",
#             "value": "usgs-01011000"
#         }
#     ]
).query(
    group_by=SIM_GROUPBY,
    order_by=SIM_GROUPBY,
    include_metrics=SIM_METRICS
).to_sdf()

In [ ]:
%%time
sdf.show()

In [ ]:
locations_sdf = ev.locations.filter("id like 'usgs-%'").to_sdf()
joined_gdf = join_geometry(sdf, locations_sdf, "primary_location_id")
joined_gdf

In [ ]:
# Make a simple plot and color stations by relative_bias.
metrics_prj = joined_gdf.to_crs("EPSG:3857")
tiles = gv.tile_sources.OSM
hvplot = metrics_prj.hvplot(
    crs=ccrs.GOOGLE_MERCATOR,
    size=75,
    c="relative_bias",
    clim=(-1,1)
)

(tiles*hvplot).opts(width=1000, height=500)

In [ ]:
SIM_GROUPBY = ["primary_location_id", "configuration_name", "variable_name", "unit_name", "event_above"]

sdf = ev.table(
    table_name="sim_joined_timeseries"
).add_calculated_fields([
    tcf.AbovePercentileEventDetection()
]).query(
    group_by=SIM_GROUPBY,
    order_by=SIM_GROUPBY,
    include_metrics=SIM_METRICS
).to_sdf()



In [ ]:
%%time
sdf.show()

In [ ]:
spark.stop()